# 팀 최고 V7 Pair-Neighbor — 독립 재현 검증

- 팀원이 제공한 최종 제출 코드의 파라미터를 변경하지 않음
- `CV seed = 42, 77, 2026`의 5-Fold 검증
- 각 Fold Train으로만 결측치·인코더·순위·이웃 탐색 학습
- 제공된 `0.146644` 수치를 사용하지 않고 직접 재계산
- 최종 Test 예측이 팀 원본 코드 출력과 일치하는지 확인

In [1]:
from __future__ import annotations

import argparse
import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.52
PAIR_QUANTILE = 0.52
PAIR_WEIGHT = 0.15

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant", fill_value="missing"
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore", sparse_output=False
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_quantile_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_test = add_feature_copies(test_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    random_state=RANDOM_SEED,
                    n_jobs=1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    matrix = pipeline.named_steps["preprocessor"].transform(weighted_test)
    forest = pipeline.named_steps["model"]
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in forest.estimators_]
    )
    return np.clip(
        np.round(np.quantile(tree_predictions, TREE_QUANTILE, axis=1), 2),
        0.0,
        1.0,
    )


def empirical_rank_transform(
    train_values: np.ndarray,
    test_values: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Train 분포만으로 Train과 한 개씩 독립적인 Test 값을 변환."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(
        sorted_train, train_values, side="right"
    ) / len(train_values)
    test_rank = np.searchsorted(
        sorted_train, test_values, side="right"
    ) / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_neighbor_prediction(
    train_features: pd.DataFrame,
    test_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    """각 2차원 피처쌍에서 가장 가까운 Train target을 분위수 집계."""
    train_rank = {}
    test_rank = {}
    for column in PAIR_COLUMNS:
        train_rank[column], test_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            test_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(test_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(test_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_index = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_index])
    return np.quantile(
        np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1
    )


def validate_inputs(
    train: pd.DataFrame,
    test: pd.DataFrame,
    submission: pd.DataFrame,
) -> None:
    if set(train.columns) - {TARGET} != set(test.columns):
        raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
    if list(submission.columns) != [ID_COLUMN, TARGET]:
        raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
    if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
        raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")
    if train[TARGET].isna().any():
        raise ValueError("Train 타깃에 결측값이 있습니다.")

## 1. 데이터 불러오기

In [2]:
DATA_DIR=Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR=Path("/Users/bitsaem/Desktop/stress_project")
CV_SEEDS=(42,77,2026)

train=pd.read_csv(DATA_DIR/"train.csv")
test=pd.read_csv(DATA_DIR/"test.csv")
submission=pd.read_csv(DATA_DIR/"sample_submission.csv")
validate_inputs(train,test,submission)

X=add_features(train.drop(columns=[TARGET,ID_COLUMN])).reset_index(drop=True)
X_test=add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)
y=train[TARGET].reset_index(drop=True)
print("Train:",X.shape,"| Test:",X_test.shape)

Train: (3000, 22) | Test: (3000, 22)


## 2. 누수 없는 Repeated 5-Fold 검증

In [3]:
summary=[]
fold_rows=[]
oof_by_seed={}

for cv_seed in CV_SEEDS:
    cv=KFold(n_splits=5,shuffle=True,random_state=cv_seed)
    oof=np.zeros(len(y))
    for fold,(train_idx,valid_idx) in enumerate(cv.split(X),start=1):
        fold_train=X.iloc[train_idx]
        fold_valid=X.iloc[valid_idx]
        fold_target=y.iloc[train_idx]
        tree_pred=tree_quantile_prediction(fold_train,fold_valid,fold_target)
        pair_pred=pair_neighbor_prediction(fold_train,fold_valid,fold_target)
        prediction=np.clip(np.round(
            (1-PAIR_WEIGHT)*tree_pred+PAIR_WEIGHT*pair_pred,2
        ),0,1)
        oof[valid_idx]=prediction
        score=mean_absolute_error(y.iloc[valid_idx],prediction)
        fold_rows.append({"CV_seed":cv_seed,"Fold":fold,"MAE":score})
        print(f"CV {cv_seed} | Fold {fold} MAE: {score:.6f}")
    cv_mae=mean_absolute_error(y,oof)
    summary.append({"CV_seed":cv_seed,"CV_MAE":cv_mae})
    oof_by_seed[cv_seed]=oof
    print(f"CV {cv_seed} overall MAE: {cv_mae:.6f}\n")

summary_frame=pd.DataFrame(summary)
fold_frame=pd.DataFrame(fold_rows)
print("Repeated-CV mean MAE:",f"{summary_frame.CV_MAE.mean():.6f}")
print("Repeated-CV MAE std:",f"{summary_frame.CV_MAE.std(ddof=0):.6f}")
display(summary_frame)
display(fold_frame)

CV 42 | Fold 1 MAE: 0.134500
CV 42 | Fold 2 MAE: 0.150167
CV 42 | Fold 3 MAE: 0.148250
CV 42 | Fold 4 MAE: 0.159217
CV 42 | Fold 5 MAE: 0.150617
CV 42 overall MAE: 0.148550

CV 77 | Fold 1 MAE: 0.149917
CV 77 | Fold 2 MAE: 0.153417
CV 77 | Fold 3 MAE: 0.147300
CV 77 | Fold 4 MAE: 0.144333
CV 77 | Fold 5 MAE: 0.136967
CV 77 overall MAE: 0.146387

CV 2026 | Fold 1 MAE: 0.141700
CV 2026 | Fold 2 MAE: 0.145900
CV 2026 | Fold 3 MAE: 0.144183
CV 2026 | Fold 4 MAE: 0.150967
CV 2026 | Fold 5 MAE: 0.140917
CV 2026 overall MAE: 0.144733

Repeated-CV mean MAE: 0.146557
Repeated-CV MAE std: 0.001563


,CV_seed,CV_MAE
0,42,0.148550
1,77,0.146387
2,2026,0.144733


,CV_seed,Fold,MAE
0,42,1,0.134500
1,42,2,0.150167
2,42,3,0.148250
3,42,4,0.159217
4,42,5,0.150617
5,77,1,0.149917
6,77,2,0.153417
7,77,3,0.147300
8,77,4,0.144333
9,77,5,0.136967


## 3. OOF 예측 저장

In [4]:
oof_frame=pd.DataFrame({
    ID_COLUMN:train[ID_COLUMN],
    TARGET:y,
    **{f"oof_seed_{seed}":pred for seed,pred in oof_by_seed.items()},
})
oof_path=OUTPUT_DIR/"oof_team_best_v7_verified.csv"
oof_frame.to_csv(oof_path,index=False,encoding="utf-8")
print("Saved:",oof_path)
oof_frame.head()

Saved: /Users/bitsaem/Desktop/stress_project/oof_team_best_v7_verified.csv


,ID,stress_score,oof_seed_42,oof_seed_77,oof_seed_2026
0,TRAIN_0000,0.63,0.48,0.53,0.51
1,TRAIN_0001,0.83,0.83,0.83,0.54
2,TRAIN_0002,0.70,0.52,0.49,0.49
3,TRAIN_0003,0.17,0.17,0.17,0.17
4,TRAIN_0004,0.36,0.50,0.41,0.45


## 4. 원본 설정으로 최종 제출 재생성

In [5]:
tree_prediction=tree_quantile_prediction(X,X_test,y)
neighbor_prediction=pair_neighbor_prediction(X,X_test,y)
pred_verified=np.clip(np.round(
    (1-PAIR_WEIGHT)*tree_prediction+PAIR_WEIGHT*neighbor_prediction,2
),0,1)

verified=submission.copy(); verified[TARGET]=pred_verified
verified_path=OUTPUT_DIR/"submit_team_best_v7_verified.csv"
verified.to_csv(verified_path,index=False,encoding="utf-8")
print("Saved:",verified_path)

reference_path=OUTPUT_DIR/"submit_team_best_original_pair_neighbor.csv"
if reference_path.exists():
    reference=pd.read_csv(reference_path)
    difference=np.abs(reference[TARGET].to_numpy()-pred_verified)
    print("Reference max abs difference:",difference.max())
    print("Reference identical:",bool(np.all(difference==0)))
else:
    print("Reference file not found; final prediction was still saved.")
verified.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_team_best_v7_verified.csv
Reference max abs difference: 0.0
Reference identical: True


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.48
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.48
8,TEST_0008,0.57
9,TEST_0009,0.88


## 해석 기준

- 발표·PPT에는 이 노트북에서 직접 계산된 `Repeated-CV mean MAE`만 사용
- `Reference identical: True`일 때 최종 제출 코드 재현 완료
- 기존 코드 주석의 `Audit3 0.146644`는 근거가 없으므로 인용하지 않음